# 05 — Hybrid Recommender

Blends normalised **popularity** and **collaborative** signals:

```
hybrid_score = POPULARITY_WEIGHT * popularity_norm
             +   COLLAB_WEIGHT   * collab_norm
```

Both components are normalised to [0, 1] before blending.

**Optimisations applied**
- Hardcoded 0.3 / 0.7 weights replaced by named constants `POPULARITY_WEIGHT` /
  `COLLAB_WEIGHT` — easy to tune without hunting through the formula.
- Filter `num_ratings >= MIN_RATINGS` at startup to pre-shrink the candidate
  pool from 87 k → ~8 k rows on every function call.
- Vectorised SVD scoring (`mu + bu + bi + qi @ pu`) — no `model.predict()` loop.
- Cold-start users fall back gracefully to pure popularity via `global_mean`.
- Dtype hints / `usecols` on CSV reads.
- `os.makedirs(exist_ok=True)` guards the outputs directory.

In [ ]:
import os
import joblib

import numpy as np
import pandas as pd

from surprise import Dataset, Reader

In [ ]:
# ── Tunable parameters ────────────────────────────────────────────────────────
POPULARITY_WEIGHT = 0.30   # weight for the popularity signal
COLLAB_WEIGHT     = 0.70   # weight for the collaborative signal
MIN_RATINGS       = 50     # minimum votes to include a movie in recommendations

assert abs(POPULARITY_WEIGHT + COLLAB_WEIGHT - 1.0) < 1e-9, "Weights must sum to 1."

In [ ]:
movies = pd.read_csv(
    "../data/movies.csv",
    dtype={"movieId": "int32"},
    encoding="utf-8",
)

ratings = pd.read_csv(
    "../data/ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32"},
    usecols=["userId", "movieId", "rating"],
    encoding="utf-8",
)

In [ ]:
# ── Popularity model ──────────────────────────────────────────────────────────
movie_stats = (
    ratings
    .groupby("movieId", sort=False)
    .agg(avg_rating=("rating", "mean"), num_ratings=("rating", "count"))
    .reset_index()
)

C = float(movie_stats["avg_rating"].mean())
m = float(movie_stats["num_ratings"].quantile(0.90))

v = movie_stats["num_ratings"]
movie_stats["popularity_score"] = (v / (v + m)) * movie_stats["avg_rating"] + (m / (v + m)) * C

In [ ]:
# ── Build hybrid candidate frame ──────────────────────────────────────────────
# Pre-filter to movies with enough ratings to reduce per-call work.
hybrid_movies = movies.merge(
    movie_stats[["movieId", "avg_rating", "num_ratings", "popularity_score"]],
    on="movieId",
    how="left",
)

hybrid_movies["popularity_score"] = hybrid_movies["popularity_score"].fillna(0.0)
hybrid_movies["avg_rating"]       = hybrid_movies["avg_rating"].fillna(0.0)
hybrid_movies["num_ratings"]      = hybrid_movies["num_ratings"].fillna(0).astype("int32")

max_score = hybrid_movies["popularity_score"].max()
hybrid_movies["popularity_norm"] = hybrid_movies["popularity_score"] / max_score if max_score > 0 else 0.0

# Pre-filter: drop movies with too few ratings once, not on every call.
hybrid_movies = hybrid_movies[hybrid_movies["num_ratings"] >= MIN_RATINGS].copy()
print(f"Hybrid candidate pool: {len(hybrid_movies):,} movies")

In [ ]:
# ── Load SVD model ────────────────────────────────────────────────────────────
model = joblib.load("../models/svd_model.pkl")
print("SVD model loaded successfully!")

In [ ]:
def hybrid_recommend(
    user_id: int,
    n: int = 10,
    popularity_weight: float = POPULARITY_WEIGHT,
    collab_weight: float = COLLAB_WEIGHT,
) -> pd.DataFrame:
    """Hybrid recommendation blending popularity + collaborative SVD signals.

    Parameters
    ----------
    user_id           : MovieLens user ID
    n                 : number of results to return
    popularity_weight : weight for the popularity signal  (default 0.30)
    collab_weight     : weight for the collaborative signal (default 0.70)
    """
    rated_ids = set(ratings.loc[ratings["userId"] == user_id, "movieId"])
    candidates = hybrid_movies[~hybrid_movies["movieId"].isin(rated_ids)].copy()

    # ── Vectorised collaborative scores ──────────────────────────────────────
    try:
        inner_uid = model.trainset.to_inner_uid(user_id)
        mu = model.trainset.global_mean
        bu = model.bu[inner_uid]
        pu = model.pu[inner_uid]

        all_scores = np.clip(
            mu + bu + model.bi + model.qi @ pu,
            0.5, 5.0,
        )

        raw_iids = np.array(
            [int(model.trainset.to_raw_iid(i)) for i in range(model.trainset.n_items)],
            dtype="int32",
        )
        score_series = pd.Series(all_scores, index=raw_iids)

        fallback = float(np.clip(mu + bu, 0.5, 5.0))
        candidates["collab_score"] = candidates["movieId"].map(score_series).fillna(fallback)

    except ValueError:
        # Cold-start: unknown user — use global mean so hybrid degrades to popularity.
        candidates["collab_score"] = float(model.trainset.global_mean)

    # ── Normalise & blend ─────────────────────────────────────────────────────
    candidates["collab_norm"] = (candidates["collab_score"] - 0.5) / 4.5

    candidates["hybrid_score"] = (
        popularity_weight * candidates["popularity_norm"]
        + collab_weight   * candidates["collab_norm"]
    )

    return (
        candidates
        .sort_values("hybrid_score", ascending=False)
        [["title", "avg_rating", "num_ratings", "hybrid_score"]]
        .head(n)
        .reset_index(drop=True)
    )

In [ ]:
hybrid_recommend(1)    # Test-1

In [ ]:
hybrid_recommend(100)  # Test-2

In [ ]:
hybrid_recommend(500)  # Test-3

In [ ]:
os.makedirs("../outputs", exist_ok=True)
hybrid_recommend(1).to_csv("../outputs/hybrid_recommendations.csv", index=False)
print("Saved → ../outputs/hybrid_recommendations.csv")